# Two-Phase YOLO26n Pipeline Runner

This notebook runs the Sprint 4 two-phase pipeline in English, using a YOLO26n weapon detector checkpoint for both:

- Stage 2 weapon detection inside approved person crops
- the single-stage baseline comparison on the full test split

Workflow:

1. Build Phase 1 person crops
2. Train the `hold` / `no_hold` classifier
3. Run two-phase inference on a split
4. Compare two-phase vs single-stage YOLO26n


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
import torch
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
assert (PROJECT_ROOT / 'scripts').exists(), 'Run this notebook from the repository root.'

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

def run_command(args):
    args = [str(arg) for arg in args]
    print('Running:')
    print(' '.join(args))
    subprocess.run(args, cwd=PROJECT_ROOT, check=True)

def show_csv(path, rows=10):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Missing file: {path}')
    df = pd.read_csv(path)
    print(f'{path} -> {len(df)} rows')
    return df.head(rows)

def show_text_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Missing file: {path}')
    display(Markdown(path.read_text(encoding='utf-8')))

def crop_outputs_exist():
    crops_root = PROJECT_ROOT / 'data' / 'interim' / 'two_phase' / 'crops'
    return any((crops_root / split_name).exists() for split_name in ['train', 'val', 'test'])


In [ ]:
# Main parameters
AUTO_DEVICE = '0' if torch.cuda.is_available() else 'cpu'
DEVICE = AUTO_DEVICE
SPLIT = 'test'
OUTPUT_PREFIX = SPLIT
OVERWRITE_CROPS = False

# Optional overrides
PERSON_MODEL = None
HOLD_CHECKPOINT = PROJECT_ROOT / 'runs' / 'two_phase' / 'carry_classifier' / 'best.pt'

# Use your YOLO26n best.pt here.
DEFAULT_WEAPON_MODEL = PROJECT_ROOT / 'runs' / 'weapon_detector' / 'weights' / 'best.pt'
WEAPON_MODEL = DEFAULT_WEAPON_MODEL if DEFAULT_WEAPON_MODEL.exists() else Path('CAMINHO_DO_BEST_PT_DO_YOLO26N')

# Optional training overrides. Keep as None to use configs/two_phase.yaml defaults.
EPOCHS = None
BATCH_SIZE = None
NUM_WORKERS = None

TWO_PHASE_PREDICTIONS = PROJECT_ROOT / 'runs' / 'two_phase' / 'predictions' / f'{OUTPUT_PREFIX}_predictions.csv'
TWO_PHASE_IMAGE_SUMMARY = PROJECT_ROOT / 'runs' / 'two_phase' / 'predictions' / f'{OUTPUT_PREFIX}_image_summary.csv'
TWO_PHASE_PIPELINE_SUMMARY = PROJECT_ROOT / 'runs' / 'two_phase' / 'predictions' / f'{OUTPUT_PREFIX}_pipeline_summary.csv'
EVAL_COMPARISON = PROJECT_ROOT / 'runs' / 'two_phase' / 'evaluation' / f'{OUTPUT_PREFIX}_comparison.csv'
EVAL_SUMMARY = PROJECT_ROOT / 'runs' / 'two_phase' / 'evaluation' / f'{OUTPUT_PREFIX}_summary.md'

print('Python executable:', sys.executable)
print('Project root:', PROJECT_ROOT)
print('torch.cuda.is_available():', torch.cuda.is_available())
print('torch.cuda.device_count():', torch.cuda.device_count())
print('Split:', SPLIT)
print('Device:', DEVICE)
print('Weapon model:', WEAPON_MODEL)


In [ ]:
required_paths = [
    PROJECT_ROOT / 'scripts' / 'build_two_phase_dataset.py',
    PROJECT_ROOT / 'scripts' / 'train_carry_classifier.py',
    PROJECT_ROOT / 'scripts' / 'run_two_phase_inference.py',
    PROJECT_ROOT / 'scripts' / 'evaluate_detection_pipeline.py',
    PROJECT_ROOT / 'configs' / 'two_phase.yaml',
    PROJECT_ROOT / 'data' / 'splits' / f'{SPLIT}_manifest.csv',
]

missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required files:\n- ' + '\n- '.join(missing))

if not Path(WEAPON_MODEL).exists():
    raise FileNotFoundError(
        'WEAPON_MODEL does not exist. Update the parameter cell with the path to your YOLO26n best.pt checkpoint.'
    )

print('All required files were found.')


## Step 1. Build the Phase 1 person crops

This generates the `hold` / `no_hold` person crops and metadata under `data/interim/two_phase/`.


In [ ]:
if crop_outputs_exist() and not OVERWRITE_CROPS:
    print('Existing Phase 1 crop outputs were found.')
    print('Skipping Step 1 to avoid overwriting previous results.')
    print('Set OVERWRITE_CROPS = True in the parameter cell if you want to rebuild them.')
else:
    cmd = [sys.executable, 'scripts/build_two_phase_dataset.py', '--device', DEVICE]

    if OVERWRITE_CROPS:
        cmd.append('--overwrite')

    if PERSON_MODEL is not None:
        cmd.extend(['--person-model', PERSON_MODEL])

    run_command(cmd)


In [ ]:
metadata_root = PROJECT_ROOT / 'data' / 'interim' / 'two_phase' / 'metadata'
for split_name in ['train', 'val', 'test']:
    crop_csv = metadata_root / f'{split_name}_person_crops.csv'
    miss_csv = metadata_root / f'{split_name}_stage0_misses.csv'
    print('\n===', split_name.upper(), '===')
    if crop_csv.exists():
        crop_df = pd.read_csv(crop_csv)
        print('crops:', len(crop_df), '| labels:', crop_df['label'].value_counts().to_dict() if not crop_df.empty else {})
    else:
        print('Missing:', crop_csv)
    if miss_csv.exists():
        miss_df = pd.read_csv(miss_csv)
        print('stage0 misses:', len(miss_df))
    else:
        print('Missing:', miss_csv)


## Step 2. Train the `hold` / `no_hold` classifier

This writes the best checkpoint to `runs/two_phase/carry_classifier/best.pt`.


In [ ]:
cmd = [sys.executable, 'scripts/train_carry_classifier.py', '--device', DEVICE]

if EPOCHS is not None:
    cmd.extend(['--epochs', str(EPOCHS)])
if BATCH_SIZE is not None:
    cmd.extend(['--batch-size', str(BATCH_SIZE)])
if NUM_WORKERS is not None:
    cmd.extend(['--num-workers', str(NUM_WORKERS)])

run_command(cmd)


In [ ]:
carry_dir = PROJECT_ROOT / 'runs' / 'two_phase' / 'carry_classifier'
metrics_path = carry_dir / 'metrics.csv'
summary_path = carry_dir / 'summary.md'
checkpoint_path = carry_dir / 'best.pt'

print('Checkpoint exists:', checkpoint_path.exists(), checkpoint_path)
display(show_csv(metrics_path, rows=12))


In [ ]:
show_text_file(PROJECT_ROOT / 'runs' / 'two_phase' / 'carry_classifier' / 'summary.md')


## Step 3. Run two-phase inference

This uses:

- the person detector from `configs/two_phase.yaml` unless you override it
- the trained `hold` / `no_hold` classifier checkpoint
- your YOLO26n `best.pt` checkpoint as the Stage 2 weapon detector


In [ ]:
cmd = [
    sys.executable,
    'scripts/run_two_phase_inference.py',
    '--split', SPLIT,
    '--device', DEVICE,
    '--weapon-model', str(WEAPON_MODEL),
    '--output-prefix', OUTPUT_PREFIX,
]

if PERSON_MODEL is not None:
    cmd.extend(['--person-model', PERSON_MODEL])

if Path(HOLD_CHECKPOINT).exists():
    cmd.extend(['--hold-checkpoint', str(HOLD_CHECKPOINT)])

run_command(cmd)


In [ ]:
print('Predictions preview')
display(show_csv(TWO_PHASE_PREDICTIONS, rows=10))

print('\nImage summary preview')
display(show_csv(TWO_PHASE_IMAGE_SUMMARY, rows=10))

print('\nPipeline summary preview')
display(show_csv(TWO_PHASE_PIPELINE_SUMMARY, rows=10))


## Step 4. Compare the two-phase pipeline against the single-stage YOLO26n baseline

The same YOLO26n checkpoint is used as the single-stage baseline model on the full image.


In [ ]:
cmd = [
    sys.executable,
    'scripts/evaluate_detection_pipeline.py',
    '--split', SPLIT,
    '--device', DEVICE,
    '--single-stage-model', str(WEAPON_MODEL),
    '--two-phase-predictions', str(TWO_PHASE_PREDICTIONS),
    '--two-phase-image-summary', str(TWO_PHASE_IMAGE_SUMMARY),
    '--output-prefix', OUTPUT_PREFIX,
]

run_command(cmd)


In [ ]:
comparison_df = pd.read_csv(EVAL_COMPARISON)
display(comparison_df)


In [ ]:
show_text_file(EVAL_SUMMARY)


## Output files

After a successful run, the main artifacts are:

- `data/interim/two_phase/crops/`
- `data/interim/two_phase/metadata/`
- `runs/two_phase/carry_classifier/best.pt`
- `runs/two_phase/predictions/{split}_predictions.csv`
- `runs/two_phase/predictions/{split}_image_summary.csv`
- `runs/two_phase/evaluation/{split}_comparison.csv`
